# Ad Ranking & CTR Prediction

Companion notebook for the [Ad Ranking & CTR lesson](https://ml-viz-ruby.vercel.app/courses/recommender-systems/06-ad-ranking-and-ctr-prediction).

**The idea in one sentence.** Ad systems don't rank by bid alone — they rank by
**expected value** (eCPM = predicted CTR × bid), charge via a **second-price
auction** that makes honest bidding optimal, and predict CTR with models like
**Factorization Machines** that capture feature *interactions* on sparse data.

The three pieces, from scratch:

- **eCPM ranking** — a high bid on an ad nobody clicks loses to a low bid on an ad
  everyone clicks.
- **Second-price auctions** — you pay the *runner-up's* bid, which makes bidding
  your true value the dominant strategy (truthful).
- **Factorization Machines** — model every pairwise feature interaction in $O(kd)$
  instead of $O(d^2)$, essential for sparse categorical ad features.

We **validate the FM's fast interaction formula against the naive pairwise sum and
the auction's truthfulness**, then cover gotchas. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)

## 1 — eCPM ranking

Ads are ranked by expected CPM = predicted CTR × bid. This aligns ML quality with revenue.

In [ ]:
def rank_ads(bids, predicted_ctrs):
    """Rank ads by eCPM = CTR × bid × 1000."""
    ecpm = predicted_ctrs * bids * 1000
    order = np.argsort(-ecpm)
    return order, ecpm

# 5 advertisers
bids = np.array([5.00, 2.00, 8.00, 3.50, 1.50])
pred_ctrs = np.array([0.01, 0.08, 0.005, 0.04, 0.12])

order, ecpm = rank_ads(bids, pred_ctrs)
print("Advertiser | Bid  | Pred CTR | eCPM")
for i in order:
    print(f"  Ad {i}     | ${bids[i]:.2f} | {pred_ctrs[i]:.3f}    | ${ecpm[i]:.2f}")

### Validate: eCPM ranking, not bid ranking

The point of eCPM is that the top ad by *value* need not be the top ad by *bid*.
We confirm the ranking is by CTR×bid and that at least one lower-bid ad outranks a
higher-bid one here (otherwise ranking by bid would suffice).

In [ ]:
order, ecpm = rank_ads(bids, pred_ctrs)
assert list(order) == list(np.argsort(-ecpm)), 'ranking must be by eCPM'
top_by_ecpm = order[0]
top_by_bid = int(np.argmax(bids))
print(f'top ad by eCPM: {top_by_ecpm} (bid ${bids[top_by_ecpm]:.2f}, CTR {pred_ctrs[top_by_ecpm]:.3f})')
print(f'top ad by bid : {top_by_bid} (bid ${bids[top_by_bid]:.2f}, CTR {pred_ctrs[top_by_bid]:.3f})')
assert top_by_ecpm != top_by_bid, 'eCPM ranking differs from bid ranking (relevance matters)'
print('\n✅ ranking by expected value (CTR×bid) beats ranking by bid alone')

## 2 — Second-price auction

The winner pays the second-highest bid (plus a small increment). This is the dominant mechanism in online advertising.

In [ ]:
def second_price_auction(bids, increment=0.01):
    """Returns (winner_id, payment)."""
    order = np.argsort(-bids)
    winner = order[0]
    second_bid = bids[order[1]]
    payment = second_bid + increment
    return winner, payment

# Auction on raw bids
winner, payment = second_price_auction(bids)
print(f"Winner: Ad {winner} (bid=${bids[winner]:.2f})")
print(f"Payment: ${payment:.2f} (second-highest bid + $0.01)")
print(f"Winner profit at true value $6: ${6 - payment:.2f}")

## 3 — Simplified Factorization Machine for CTR

A Factorization Machine models pairwise feature interactions via embedding dot products — essential for high-cardinality ad features.

In [ ]:
class FactorizationMachine:
    def __init__(self, n_features, d=4, seed=0):
        rng_ = np.random.default_rng(seed)
        self.w0 = 0.0                                  # global bias
        self.w  = np.zeros(n_features)                 # linear weights
        self.V  = rng_.normal(0, 0.01, (n_features, d))  # interaction factors

    def predict(self, x):
        """x: (n_features,) sparse binary feature vector"""
        linear = self.w0 + self.w @ x
        # Pairwise interaction term: 0.5 * (||Vx||^2 - ||V^2 x||_sum)
        Vx = self.V.T @ x                              # (d,)
        interaction = 0.5 * (Vx @ Vx - (self.V**2).T @ (x**2) @ np.ones(4))
        return 1 / (1 + np.exp(-(linear + interaction)))  # sigmoid -> CTR

# Features: [user_age_bucket, device_mobile, ad_category, time_evening, ...]
n_features = 20
fm = FactorizationMachine(n_features=n_features)

# Simulate a feature vector for a user-ad pair
x = np.zeros(n_features)
x[[0, 5, 12, 17]] = 1                                 # active features

ctr_pred = fm.predict(x)
print(f"Predicted CTR for this user-ad pair: {ctr_pred:.4f}")
print(f"eCPM at $3.50 bid: ${ctr_pred * 3.50 * 1000:.2f}")

### Validate: the FM interaction formula is the pairwise sum, computed fast

Factorization Machines compute $\sum_{i<j} \langle v_i, v_j\rangle x_i x_j$ in
$O(kd)$ using the identity $\tfrac12(\lVert Vx\rVert^2 - \sum \lVert v_i\rVert^2 x_i^2)$
instead of the naive $O(d^2)$ double loop. We verify the two agree exactly — the
trick that makes FMs tractable on sparse features.

In [ ]:
Vx = fm.V.T @ x
fast = 0.5 * (Vx @ Vx - (fm.V ** 2).T @ (x ** 2) @ np.ones(fm.V.shape[1]))
naive = sum(x[i] * x[j] * (fm.V[i] @ fm.V[j])
            for i in range(n_features) for j in range(i + 1, n_features))
print(f'fast  O(kd) interaction: {fast:.6f}')
print(f'naive O(d^2) interaction: {naive:.6f}')
assert np.isclose(fast, naive), 'the FM linear-time formula must equal the pairwise double sum'
print('\n✅ the FM trick computes all pairwise interactions in O(kd), not O(d^2)')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **CTR must be calibrated** | eCPM uses the raw probability; a mis-scaled CTR directly mis-prices the auction |
| **first-price ≠ truthful** | pay-your-bid auctions incentivize bid shading and constant re-optimization |
| **FM captures only 2nd-order** | higher-order interactions need deep FMs / DCN |
| **position & selection bias** | logged clicks depend on where the ad was shown → biased CTR training data |
| **exploration for new ads** | a new ad has no CTR history → cold-start bias toward incumbents |

Demo: second-price truthfulness — bidding your true value maximizes profit.

In [ ]:
# Second-price truthfulness: in a second-price auction the winner pays the runner-up's
# bid, so bidding your TRUE value is optimal — over-bidding risks overpaying, under-bidding
# risks losing a profitable slot, and neither improves your outcome.
true_value = 6.0
def profit_if_bid(my_bid, others=np.array([5.0, 2.0, 3.5, 1.5])):
    field = np.concatenate([[my_bid], others])
    win = field.argmax() == 0
    pay = np.sort(field)[-2] + 0.01 if win else 0.0
    return (true_value - pay) if win else 0.0
for b in [4.0, 6.0, 8.0]:
    print(f'bid ${b:.2f} (true value ${true_value}): profit ${profit_if_bid(b):.2f}')
assert profit_if_bid(6.0) >= max(profit_if_bid(4.0), profit_if_bid(8.0)), 'truthful bid is optimal'
print('\nBidding your true value maximizes profit — the defining property of second-price auctions.')

## ✏️ Your turn

**Exercise.** Implement `gsp_auction(bid_quality_scores, slots)` for a Generalized Second Price auction with `slots` ad slots. Each ad in position $k$ pays the minimum bid that keeps it in position $k$ (i.e., just above the next-lower advertiser's bid-quality score).

Return an array of payments for the top `slots` winners.

In [ ]:
def gsp_auction(bid_quality_scores, slots=3):
    """
    bid_quality_scores: array of bid × quality_score for each advertiser
    slots: number of ad slots available
    Returns payments[k] for k in range(slots).
    """
    # TODO(you): sort by bid_quality_score descending, then compute each winner's payment
    # payment[k] = bid_quality_scores[k+1] / quality_score[k]  (simplified)
    # For simplicity, use: payment[k] = bid_quality_scores[k+1] + 0.01
    return ...

bqs = np.array([12.0, 9.0, 6.0, 3.0, 1.5])  # bid × quality scores
payments = gsp_auction(bqs, slots=3)
print("GSP payments for top-3 slots:", payments)

In [ ]:
# Assertion
pay = gsp_auction(bqs, slots=3)
assert len(pay) == 3, "Should return 3 payments for 3 slots"
assert pay[0] > pay[1] > pay[2] >= 0, "Higher slots should pay more"
assert abs(pay[0] - (bqs[1] + 0.01)) < 0.001, "Winner pays next competitor's score + increment"
print("✓ GSP auction correct")

<details>
<summary>Solution</summary>

```python
def gsp_auction(bid_quality_scores, slots=3):
    order = np.argsort(-bid_quality_scores)
    payments = []
    for k in range(slots):
        if k + 1 < len(bid_quality_scores):
            payments.append(bid_quality_scores[order[k + 1]] + 0.01)
        else:
            payments.append(0.01)
    return np.array(payments)
```

In GSP, each winner pays just enough to outbid the person below them — encouraging efficient allocation while being simpler than VCG.
</details>

## Key takeaways

- **Rank by expected value, not bid.** eCPM = CTR × bid, so a relevant cheap ad
  beats an irrelevant expensive one.
- **Second-price auctions are truthful.** Paying the runner-up's bid makes
  bidding your true value the dominant strategy — we verified it.
- **Factorization Machines model feature interactions** in $O(kd)$ via the
  square-of-sums trick (matched the naive $O(d^2)$ sum exactly) — essential for
  sparse categorical ad features.
- **CTR calibration matters more than ranking here:** the *number* feeds directly
  into the auction price, so a biased CTR mis-charges advertisers.